<a href="https://colab.research.google.com/github/Rustam99-eng/Convolutional-Neural-Networks-Cats-and-Dogs/blob/main/convolutional_neural_networks%2C_cats_and_dogs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Импортируем библиотеки
import keras
import numpy as np
import os
import shutil
from keras import layers, models, optimizers
from keras.applications import MobileNet
from keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from keras import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
# Загрузим датасет
!wget https://storage.yandexcloud.net/academy.ai/cat-and-dog.zip

--2026-03-29 16:23:51--  https://storage.yandexcloud.net/academy.ai/cat-and-dog.zip
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228082266 (218M) [application/x-zip-compressed]
Saving to: ‘cat-and-dog.zip’

cat-and-dog.zip     100%[===================>] 217.52M  13.1MB/s    in 18s     

2026-03-29 16:24:10 (11.9 MB/s) - ‘cat-and-dog.zip’ saved [228082266/228082266]



In [3]:
# Разархивируем датасет во временную папку 'temp'
!unzip -qo "cat-and-dog" -d ./temp

# Папка с папками картинок, рассортированных по категориям
IMAGE_PATH = './temp/training_set/training_set/'

# Папка в которой будем создавать выборки
BASE_DIR = './dataset/'

In [4]:
# Проверим содержимое папки
import os
os.listdir(IMAGE_PATH)

['dogs', 'cats']

In [5]:
# Определение списка имен классов
CLASS_LIST = sorted(os.listdir(IMAGE_PATH))

# Определение количества классов
CLASS_COUNT = len(CLASS_LIST)

# Проверка результата
print(f'Количество классов: {CLASS_COUNT}, метки классов: {CLASS_LIST}')

Количество классов: 2, метки классов: ['cats', 'dogs']


In [6]:
data_files = []  # Cписок путей к файлам изображений
data_labels = [] # Список меток классов

for class_label in range(CLASS_COUNT):    # Перебор по всем классам по порядку номеров (их меток)
    class_name = CLASS_LIST[class_label]  # Выборка имени класса из списка имен
    class_path = IMAGE_PATH + class_name  # Полный путь к папке с изображениями класса

    # Получение списка имен файлов с изображениями текущего класса
    class_files = os.listdir(class_path)

    # Вывод информации о численности класса
    print(f'Размер класса {class_name} составляет {len(class_files)} животных')

    # Добавление к общему списку всех файлов класса с добавлением родительского пути
    data_files += [f'{class_path}/{file_name}' for file_name in class_files]

    # Добавление к общему списку меток текущего класса - их ровно столько, сколько файлов в классе
    data_labels += [class_label] * len(class_files)

print('Общий размер базы для обучения:', len(data_labels))

Размер класса cats составляет 4000 животных
Размер класса dogs составляет 4005 животных
Общий размер базы для обучения: 8005


In [7]:
# Набор утилит для работы с файловой системой
import shutil

# При повторном запуске пересоздаим структуру каталогов
# Если папка существует, то удаляем ее со всеми вложенными каталогами и файлами
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)

# Создаем папку по пути BASE_DIR
os.mkdir(BASE_DIR)

# Сцепляем путь до папки с именем вложенной папки. Аналогично BASE_DIR + 'train'
train_dir = os.path.join(BASE_DIR, 'train')

# Создаем подпапку, используя путь
os.mkdir(train_dir)

# Сцепляем путь до папки с именем вложенной папки. Аналогично BASE_DIR + 'validation'
validation_dir = os.path.join(BASE_DIR, 'validation')

# Создаем подпапку, используя путь
os.mkdir(validation_dir)

# Сцепляем путь до папки с именем вложенной папки. Аналогично BASE_DIR + 'test'
test_dir = os.path.join(BASE_DIR, 'test')

# Создаем подпапку, используя путь
os.mkdir(test_dir)

In [8]:
# Функция создания подвыборок (папок с файлами)
def create_dataset(
    img_path: str,         # Путь к файлам с изображениями классов
    new_path: str,         # Путь к папке с выборками
    class_name: str,       # Имя класса (оно же и имя папки)
    start_index: int,      # Стартовый индекс изображения, с которого начинаем подвыборку
    end_index: int         # Конечный индекс изображения, до которого создаем подвыборку

):

    src_path = os.path.join(img_path, class_name)  # Полный путь к папке с изображениями класса
    dst_path = os.path.join(new_path, class_name)  # Полный путь к папке с новым датасетом класса

    # Получение списка имен файлов с изображениями текущего класса
    class_files = os.listdir(src_path)

    # Создаем подпапку, используя путь
    os.mkdir(dst_path)

    # Перебираем элементы, отобранного списка с начального по конечный индекс
    for fname in class_files[start_index : end_index]:
        # Путь к файлу (источник)
        src = os.path.join(src_path, fname)
        # Новый путь расположения файла (назначение)
        dst = os.path.join(dst_path, fname)
        # Копируем файл из источника в новое место (назначение)
        shutil.copyfile(src, dst)

In [9]:
for class_label in range(CLASS_COUNT):    # Перебор по всем классам по порядку номеров (их меток)
    class_name = CLASS_LIST[class_label]  # Выборка имени класса из списка имен

    # Создаем обучающую выборку для заданного класса из диапазона (0-2000)
    create_dataset(IMAGE_PATH, train_dir, class_name, 0, 2000)
    # Создаем проверочную выборку для заданного класса из диапазона (2000-3000)
    create_dataset(IMAGE_PATH, validation_dir, class_name, 2000, 3000)
    # Создаем тестовую выборку для заданного класса из диапазона (3000-4005)
    create_dataset(IMAGE_PATH, test_dir, class_name, 3000, 4005)

In [10]:
print('Число кошек %s, число собак %s в обучающей выборке' \
      % (
          len(os.listdir(os.path.join(train_dir, 'cats'))),
          len(os.listdir(os.path.join(train_dir, 'dogs')))
         )
      )

print('Число кошек %s, число собак %s в проверочной выборке' \
      % (
          len(os.listdir(os.path.join(validation_dir, 'cats'))),
          len(os.listdir(os.path.join(validation_dir, 'dogs')))
         )
      )

print('Число кошек %s, число собак %s в контрольной выборке' \
      % (
          len(os.listdir(os.path.join(test_dir, 'cats'))),
          len(os.listdir(os.path.join(test_dir, 'dogs')))
         )
      )

Число кошек 2000, число собак 2000 в обучающей выборке
Число кошек 1000, число собак 1000 в проверочной выборке
Число кошек 1000, число собак 1005 в контрольной выборке


In [11]:
# Функция создания модели
def model_maker(IMG_WIDTH, IMG_HEIGHT, NUM_CLASSES):
    base_model = MobileNet(include_top=False, input_shape = (IMG_WIDTH, IMG_HEIGHT, 3))

    for layer in base_model.layers[:]:
        layer.trainable = False

    input = Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3))
    custom_model = base_model(input)
    custom_model = GlobalAveragePooling2D()(custom_model)
    custom_model = Dense(64, activation='relu')(custom_model)
    custom_model = Dropout(0.5)(custom_model)
    predictions = Dense(NUM_CLASSES, activation='softmax')(custom_model)

    return Model(inputs=input, outputs=predictions)

In [12]:
# компиляция модели
model = model_maker(150, 150, CLASS_COUNT)
model.compile(loss='binary_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=2e-5),
    metrics=['acc']
)

/tmp/ipykernel_637/3378879393.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNet(include_top=False, input_shape = (IMG_WIDTH, IMG_HEIGHT, 3))


17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [13]:
#создание генераторов данных с аугментацией для тренировочного набора
train_datagen = ImageDataGenerator(
    rescale=1./255, #нормализация пикселей [0,255] -> [0,1]
    rotation_range=20, #cлучайный поворот
    width_shift_range=0.2, #cлучайный сдвиг по ширине
    height_shift_range=0.2, #cлучайный сдвиг по высоте
    shear_range=0.2, #cлучайный сдвиг (искажение)
    zoom_range=0.2, #cлучайное масштабирование [0.8, 1.2]
    horizontal_flip=True, #cлучайное зеркальное отражение
    fill_mode='nearest' #заполнение новых пикселей при трансформациях
)

#без аугментации для валидации и теста
validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='categorical'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='categorical'
)

Found 4000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
Found 2005 images belonging to 2 classes.


In [14]:
callback = keras.callbacks.EarlyStopping(
                    monitor='val_acc',
                    mode='max',
                    verbose=1,
                    min_delta=0.001,
                    patience = 5)

# Обучаем модель
history = model.fit(
    train_generator,
    epochs=30,
    validation_data=validation_generator,
)

Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 52s 188ms/step - acc: 0.6777 - loss: 0.7589 - val_acc: 0.8795 - val_loss: 0.3517
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 29s 146ms/step - acc: 0.7887 - loss: 0.4998 - val_acc: 0.9220 - val_loss: 0.2523
Epoch 3/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - acc: 0.8508 - loss: 0.3859 - val_acc: 0.9350 - val_loss: 0.2056
Epoch 4/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 151ms/step - acc: 0.8702 - loss: 0.3355 - val_acc: 0.9410 - val_loss: 0.1767
Epoch 5/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 31s 154ms/step - acc: 0.8888 - loss: 0.2873 - val_acc: 0.9475 - val_loss: 0.1576
Epoch 6/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 31s 156ms/step - acc: 0.9028 - loss: 0.2521 - val_acc: 0.9495 - val_loss: 0.1431
Epoch 7/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 31s 156ms/step - acc: 0.8960 - loss: 0.2599 - val_acc: 0.9495 - val_loss: 0.1349
Epoch 8/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 32s 161ms/step - acc: 0.9107 - loss: 0.2364 - val_acc: 0.9520 - val_loss: 0.1290
Epoch 9/30
200/200 ━━━━━━━━━━━━━

In [15]:
#точность контрольной выборки
test_samples = test_generator.samples
test_loss, test_acc = model.evaluate(test_generator, steps=test_samples // 20)
print(f'Точность контрольной выборки: {test_acc}')

100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - acc: 0.9710 - loss: 0.0706
Точность контрольной выборки: 0.9710000157356262
